# 5장 메타분석 (R과 파이썬으로 하는 통계분석)

보건통계학 기초 19장의 가상 메타분석(급성관상동맥증후군 시험 10개)을 R의 meta 패키지로 합치고, 이질성, 하위군, 메타회귀, 출판 편향, 민감도 분석, NNT, 네트워크 메타분석까지 차례로 구합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요. meta 패키지가 없으면 먼저 install.packages("meta")를 실행합니다.


## a. 메타분석 자료 준비

### 셀 1. 시험 하나가 한 줄인 자료 불러오기

In [ ]:
library(meta)
d <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/acs_trials.csv")
d$stemi10 <- d$stemi_pct / 10        # 메타회귀에 쓸 10%p 단위
d[, c("trial", "phase", "stemi_pct", "rob", "n_x", "mace_x", "n_pbo", "mace_pbo")]

### 셀 2. 오즈비와 신뢰구간만 있는 연구 넣기

In [ ]:
o <- data.frame(study = paste("Study", 1:7),
                or = c(1.15, 1.90, 1.45, 2.60, 1.05, 1.70, 3.10),
                lo = c(0.95, 1.55, 1.10, 1.90, 0.88, 1.30, 2.05),
                hi = c(1.39, 2.33, 1.91, 3.56, 1.25, 2.22, 4.69))
mo <- metagen(log(or), lower = log(lo), upper = log(hi), studlab = study,
              data = o, sm = "OR", method.tau = "REML",
              method.random.ci = "HK", prediction = TRUE, method.predict = "HK")
data.frame(study = o$study, logOR = round(mo$TE, 3), SE = round(mo$seTE, 3))

## b. 두 모형으로 합치기와 방법 고르기

### 셀 3. 계획서대로 합치기: REML, HKSJ 구간, 예측구간

In [ ]:
m <- metabin(mace_x, n_x, mace_pbo, n_pbo, studlab = trial, data = d,
             sm = "OR", method.tau = "REML", method.random.ci = "HK",
             prediction = TRUE, method.predict = "HK")
summary(m)

### 셀 4. 포레스트 플롯

In [ ]:
forest(m, sortvar = TE, prediction = TRUE, print.tau2 = TRUE,
       leftcols = c("studlab", "event.e", "n.e", "event.c", "n.c"),
       leftlabs = c("Trial", "Events", "Total", "Events", "Total"),
       label.e = "Drug X", label.c = "Placebo",
       label.left = "Favours drug X", label.right = "Favours placebo")

### 셀 5. 기본값으로 돌리면 무엇이 달라지나

In [ ]:
m0 <- metabin(mace_x, n_x, mace_pbo, n_pbo, studlab = trial, data = d,
              sm = "OR", prediction = TRUE)            # 기본값 그대로
c(tau = m0$method.tau, ci = m0$method.random.ci, predict = m0$method.predict)
round(exp(c(lower = m0$lower.random, upper = m0$upper.random,
            pi.lower = m0$lower.predict, pi.upper = m0$upper.predict)), 3)
c(P.default = m0$pval.random, P.plan = m$pval.random)

### 셀 6. 관찰연구 7개를 여러 방법으로 합치기

In [ ]:
grid <- expand.grid(tau2 = c("DL", "PM", "REML"), ci = c("classic", "HK"),
                    stringsAsFactors = FALSE)
res <- t(sapply(seq_len(nrow(grid)), function(i) {
  x <- update(mo, method.tau = grid$tau2[i], method.random.ci = grid$ci[i])
  c(OR = exp(x$TE.random), lower = exp(x$lower.random),
    upper = exp(x$upper.random), P = x$pval.random, tau2 = x$tau2)
}))
cbind(grid, round(res, 4))
round(exp(c(common = mo$TE.common, lower = mo$lower.common, upper = mo$upper.common)), 2)

### 셀 7. q가 1보다 작을 때의 수정판

In [ ]:
ma <- metabin(mace_x, n_x, mace_pbo, n_pbo, studlab = trial, data = d,
              sm = "OR", method.tau = "REML", method.random.ci = "HK",
              adhoc.hakn.ci = "se")      # q가 1보다 작으면 원래 표준오차를 씀
round(exp(c(OR = ma$TE.random, lower = ma$lower.random, upper = ma$upper.random)), 3)
round(ma$pval.random, 4)

## c. 이질성, 하위군, 메타회귀

### 셀 8. 이질성 지표를 꺼내 보기

In [ ]:
round(c(Q = m$Q, df = m$df.Q, P = m$pval.Q), 4)
round(c(I2 = m$I2, lower = m$lower.I2, upper = m$upper.I2), 3)
round(c(tau2 = m$tau2, lower = m$lower.tau2, upper = m$upper.tau2, tau = m$tau), 4)
round(exp(c(pi.lower = m$lower.predict, pi.upper = m$upper.predict)), 3)
round(update(m, method.I2 = "tau2")$I2, 3)              # I²를 τ²로 구하면
sapply(c("DL", "PM", "REML"), function(t) round(update(m, method.tau = t)$tau2, 4))

### 셀 9. 진단별 하위군과 하위군 간 차이 검정

In [ ]:
sg <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/acs_trials_subgroup.csv")
ms <- metabin(mace_x, n_x, mace_pbo, n_pbo, studlab = trial, data = sg,
              sm = "OR", method.tau = "REML", method.random.ci = "HK",
              subgroup = subgroup, common = FALSE)
summary(ms)

### 셀 10. τ²가 0인 하위군은 보통의 방법으로

In [ ]:
st <- metabin(mace_x, n_x, mace_pbo, n_pbo, studlab = trial,
              data = subset(sg, subgroup == "STEMI"), sm = "OR",
              method.tau = "REML", method.random.ci = "classic", common = FALSE)
round(c(tau2 = st$tau2, exp(c(OR = st$TE.random, lower = st$lower.random,
                               upper = st$upper.random))), 3)

### 셀 11. STEMI 비율로 메타회귀

In [ ]:
mr <- metareg(m, ~ stemi10)
mr
round(exp(c(ratio = coef(mr)[["stemi10"]], lower = mr$ci.lb[2], upper = mr$ci.ub[2])), 3)

## d. 출판 편향, 민감도 분석, NNT

### 셀 12. 깔때기 그림

In [ ]:
funnel(m)

### 셀 13. 비대칭 검정

In [ ]:
metabias(m)                     # 오즈비의 기본: Harbord 검정
round(sapply(c("Egger", "Harbord", "Peters"),
             function(b) metabias(m, method.bias = b)$pval), 3)

### 셀 14. 한 시험씩 빼기와 그 밖의 민감도 분석

In [ ]:
inf <- metainf(m, pooled = "random")
round(exp(range(inf$TE)), 3)                              # 한 시험씩 뺀 10번의 범위
low <- update(m, subset = rob == "low")                   # 편향 위험 '낮음'만
rr  <- update(m, sm = "RR")                               # 상대위험도로
est <- function(x) exp(c(OR = x$TE.random, lower = x$lower.random, upper = x$upper.random))
round(rbind(low_rob = est(low), risk_ratio = est(rr)), 3)

### 셀 15. NNT로 옮기기

In [ ]:
nnt(m, p.c = 0.084)             # p.c: 대조군의 위험(기저 위험)

## e. 네트워크 메타분석

### 셀 16. 네트워크 자료 불러오기

In [ ]:
library(netmeta)
d2 <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch19f_network.csv")
d2[c(1, 11, 16, 18), ]                 # 비교마다 한 줄씩 보기
table(paste(d2$t1, "vs", d2$t0))      # 비교마다 시험 수

### 셀 17. 네트워크 전체를 한 번에 합치기

In [ ]:
p  <- pairwise(treat = list(t1, t0), event = list(e1, e0), n = list(n1, n0),
               studlab = study, data = d2, sm = "OR")
nm <- netmeta(p, reference.group = "P", method.tau = "REML", common = FALSE)
nm
c(tau = netmeta(p, reference.group = "P")$method.tau)   # 적지 않으면

### 셀 18. 직접 근거와 간접 근거 나눠 보기

In [ ]:
ns <- netsplit(nm)
data.frame(comparison = ns$random$comparison, k = ns$k,
           direct.share = round(ns$prop.random, 2),
           direct = round(exp(ns$direct.random$TE), 2),
           indirect = round(exp(ns$indirect.random$TE), 2),
           P.diff = round(ns$compare.random$p, 2))

### 셀 19. 순위(P-score)

In [ ]:
netrank(nm, small.values = "desirable")      # 작은 오즈비가 좋은 결과

### 셀 20. 리그 표

In [ ]:
netleague(nm, common = FALSE, digits = 2, bracket = "(", separator = "-",
          seq = c("Z", "X", "Y", "P"))

### 셀 21. 네트워크 그림

In [ ]:
netgraph(nm, number.of.studies = TRUE, thickness = "number.of.studies",
         plastic = FALSE, col = "gray60", offset = 0.06)

## f. 과제 정답

### 셀 22. 과제 1 정답: 시험 단계로 나눈 하위군

In [ ]:
mp <- update(m, subgroup = paste("Phase", phase), method.random.ci = "classic")
round(exp(c(phase2 = mp$TE.random.w[["Phase 2"]], phase3 = mp$TE.random.w[["Phase 3"]])), 2)
round(c(Q = mp$Q.b.random, P = mp$pval.Q.b.random), 3)
round(update(mp, method.random.ci = "HK")$pval.Q.b.random, 3)   # HKSJ 표준오차로 검정하면

### 셀 23. 과제 2 정답: 고정효과로 바꾸면

In [ ]:
mf <- update(m, common = TRUE, random = FALSE)
round(exp(c(common = mf$TE.common, lower = mf$lower.common, upper = mf$upper.common)), 2)
round(exp(c(random = m$TE.random, lower = m$lower.random, upper = m$upper.random)), 2)